In [1]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [2]:
#DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
DATA_PATH = "D:/Google Drive/Work/ITMO/data/gbm122/pVACseq/"
METHOD = "mhcseqnet"

In [3]:
from data_reading import read_gbm122_data

per_region_total_df , per_region_per_allele_mt_df, per_region_per_allele_wt_df = read_gbm122_data(DATA_PATH)


Region 1, unique (Transcrpit, Epitope pairs): 13197
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*35:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*40:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*03:04: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*04:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
Region 2, unique (Transcrpit, Epitope pairs): 13273
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 5127
allele HLA-B*35:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 51

D:\Tools\Python\Anaconda\envs\mhcseqnet\lib\site-packages\IPython\core\interactiveshell.py:3263: DtypeWarning: Columns (10) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


Region 3, unique (Transcrpit, Epitope pairs): 15170
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*35:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*40:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*03:04: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*04:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
Region 4, unique (Transcrpit, Epitope pairs): 13757
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 4932
allele HLA-B*35:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 49

# 0. Preparations

In [4]:
sample_region_dict = next(iter(per_region_per_allele_wt_df.values()))
sample_allele_df =  next(iter(sample_region_dict.values()))

ALLELES = list(sample_region_dict.keys())
REGIONS = list(per_region_total_df.keys())
ALLELES, REGIONS

(['HLA-A*03:01', 'HLA-B*35:01', 'HLA-B*40:01', 'HLA-C*03:04', 'HLA-C*04:01'],
 [1, 2, 3, 4, 5])

## 0.1. make result dicts

In [5]:
per_region_per_method_per_allele_wt_predictions = dict()
per_region_per_method_per_allele_mt_predictions = dict()
for region in per_region_total_df.keys():
    per_region_per_method_per_allele_wt_predictions[region] = dict()
    per_region_per_method_per_allele_mt_predictions[region] = dict()

# 1. Predict with MHCSeqNet

In [6]:
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import pandas as pd
import os
import sys
import numpy as np
PREDICTOR_HOME = os.getenv('MHCSEQNET_HOME')
PREDICTOR_HOME = "D:\Projects\MHCSeqNet"

In [7]:
from MHCSeqNet.PredictionModel.BindingSequencePredictor import BindingSequencePredictor

In [8]:
# Initialize model instance
bindingSequencePredictor = BindingSequencePredictor()

In [9]:
# Load model from path. Please replace [path to project] to the path to the root of this project in your machine.
#get env variable
#'./PretrainedModels/sequence_model/'
bindingSequencePredictor.load_model(f'{PREDICTOR_HOME}/PretrainedModels/sequence_model/')

Using Sequence Model


In [29]:
def defineClass(allele_name):
    return "I"


def filter_df(df, allele_name):
    result_df = df.copy()
    if defineClass(allele_name) == 'I':
        result_df = result_df[result_df['length'] <= 15]
    else:
        result_df = result_df[result_df['length'] <= 30]
    result_df = result_df[result_df['peptide'].notnull()]
        
    return result_df
        
    
    
def predict_with_mhcseqnet(per_allele_df):
    per_allele_predictions = dict()
    for allele_name in ALLELES:
        print(f"-- allele {allele_name}")
        result_df = per_allele_df[allele_name].copy()
        target_allele = normalize_allele_name(allele_name)
        result_df['mhc'] = target_allele
        
        assert "peptide" in result_df
        assert "length" in result_df
        assert "mhc" in result_df
        
        initial_length = len(result_df)
        input_df = filter_df(result_df, allele_name) # return new filtered copy of df
        filtered_length = len(input_df)
        print(f"Initial length - {initial_length}, after filtering {filtered_length}")
        
        data = input_df[['peptide', 'mhc']].to_numpy(dtype=str)
        result = bindingSequencePredictor.predict(peptides=data[:, 0],
                                          alleles=data[:, 1])
        result_df['score'] = np.nan
        result_df.loc[input_df.index,'score'] = result # set result to those rows which were filtered
        per_allele_predictions[allele_name] = result_df
    return per_allele_predictions
    

In [31]:
for region in REGIONS:
    print(f"-region {region}")
    print(f"--mt")

    per_allele_mt_predictions = predict_with_mhcseqnet(per_region_per_allele_mt_df[region])
    per_region_per_method_per_allele_mt_predictions[region][METHOD] = per_allele_mt_predictions  

    print(f"--wt")
    per_allele_wt_predictions = predict_with_mhcseqnet(per_region_per_allele_wt_df[region])
    per_region_per_method_per_allele_wt_predictions[region][METHOD] = per_allele_wt_predictions  

-region 1
--mt
-- allele HLA-A*03:01
Initial length - 5409, after filtering 5409
Starting Prediction
Predict from model 0
170/170 - 6s
Predict from model 1
170/170 - 6s
Predict from model 2
170/170 - 6s
Predict from model 3
170/170 - 6s
Predict from model 4
170/170 - 6s
-- allele HLA-B*35:01
Initial length - 5409, after filtering 5409
Starting Prediction
Predict from model 0
170/170 - 5s
Predict from model 1
170/170 - 6s
Predict from model 2
170/170 - 5s
Predict from model 3
170/170 - 6s
Predict from model 4
170/170 - 6s
-- allele HLA-B*40:01
Initial length - 5409, after filtering 5409
Starting Prediction
Predict from model 0
170/170 - 5s
Predict from model 1
170/170 - 5s
Predict from model 2
170/170 - 5s
Predict from model 3
170/170 - 5s
Predict from model 4
170/170 - 5s
-- allele HLA-C*03:04
Initial length - 5409, after filtering 5409
Starting Prediction
Predict from model 0
170/170 - 5s
Predict from model 1
170/170 - 5s
Predict from model 2
170/170 - 5s
Predict from model 3
170/170 

In [34]:
per_region_per_method_per_allele_wt_predictions[region][METHOD][ALLELES[4]]

,allele,length,peptide,source_sequence_name,mhc,score
14936,HLA-C*04:01,8,PPRTRNKF,seq14936,HLA-C*04:01,0.855270
14939,HLA-C*04:01,8,SWPGTPVY,seq14939,HLA-C*04:01,0.970322
14940,HLA-C*04:01,8,YVAVCRPL,seq14940,HLA-C*04:01,0.962414
14941,HLA-C*04:01,8,RGLLSLAF,seq14941,HLA-C*04:01,0.765177
14942,HLA-C*04:01,8,NaN,seq14942,HLA-C*04:01,NaN
...,...,...,...,...,...,...
65802,HLA-C*04:01,11,LRGTLQKDGWT,seq65802,HLA-C*04:01,0.999874
65806,HLA-C*04:01,11,SSSSSSNNNNY,seq65806,HLA-C*04:01,0.995421
65807,HLA-C*04:01,11,RSSSSSSSSNN,seq65807,HLA-C*04:01,0.997723
65808,HLA-C*04:01,11,SSSSSSSSNNN,seq65808,HLA-C*04:01,0.999945


# 2. Join predictions

In [35]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity'],
    'acme':['source_sequence_name','score', ],
    'puffin':['source_sequence_name', 'score', 'epistemic_var', 'aleatoric_var', 'binding_likelihood'],
    'mhcseqnet':['source_sequence_name', 'score']
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df
    

def getPredictionResult(total_results, region, allele_name, method):
    df = total_results[region][method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df


In [36]:
def joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions):
    per_region_per_allele_result = dict()
    for region in REGIONS:
        print("region", region)
        per_region_per_allele_result[region] = dict()
        methods = list(per_region_per_method_per_allele_mt_predictions[region].keys())
        for allele_name in ALLELES:
            print("allele", allele_name)
            df_source_wt = per_region_per_allele_wt_df[region][allele_name].copy()
            df_source_mt = per_region_per_allele_mt_df[region][allele_name].copy()
            
            for method in methods:
                print("method", method)
                df_method_wt = getPredictionResult(per_region_per_method_per_allele_wt_predictions, region, allele_name, method)
                df_method_mt = getPredictionResult(per_region_per_method_per_allele_mt_predictions, region, allele_name, method)
                df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
                df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
            leave_columns = [ 'source_sequence_name', 'allele', 'length']
            df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
            df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
            # rename id back 

            
            df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
            per_region_per_allele_result[region][allele_name] = df_result
    return per_region_per_allele_result
                
                

              

In [37]:
per_region_per_allele_result = joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions)

region 1
allele HLA-A*03:01
method mhcseqnet
allele HLA-B*35:01
method mhcseqnet
allele HLA-B*40:01
method mhcseqnet
allele HLA-C*03:04
method mhcseqnet
allele HLA-C*04:01
method mhcseqnet
region 2
allele HLA-A*03:01
method mhcseqnet
allele HLA-B*35:01
method mhcseqnet
allele HLA-B*40:01
method mhcseqnet
allele HLA-C*03:04
method mhcseqnet
allele HLA-C*04:01
method mhcseqnet
region 3
allele HLA-A*03:01
method mhcseqnet
allele HLA-B*35:01
method mhcseqnet
allele HLA-B*40:01
method mhcseqnet
allele HLA-C*03:04
method mhcseqnet
allele HLA-C*04:01
method mhcseqnet
region 4
allele HLA-A*03:01
method mhcseqnet
allele HLA-B*35:01
method mhcseqnet
allele HLA-B*40:01
method mhcseqnet
allele HLA-C*03:04
method mhcseqnet
allele HLA-C*04:01
method mhcseqnet
region 5
allele HLA-A*03:01
method mhcseqnet
allele HLA-B*35:01
method mhcseqnet
allele HLA-B*40:01
method mhcseqnet
allele HLA-C*03:04
method mhcseqnet
allele HLA-C*04:01
method mhcseqnet


# 5. For now just write to files

In [38]:
#DATA_PATH = "/home/denklewer/Data/gbm122/" #ubuntu machine
DATA_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/gbm122/" # windows

TARGET_PATH_TO_PREDICTIONS = DATA_PATH + '{}/'.format(METHOD)

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [43]:
# for decoy_key in DECOY_PREFIXES:

for region in REGIONS:
    for allele_name in ALLELES:
        total_df = per_region_per_allele_result[region][allele_name]
        allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        total_df.to_csv("{}{}_predictions_reg_{}_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,region,allele_name),sep=";", index=False)

In [44]:
total_df

,allele,length,peptide_mt,source_sequence_name,mhcseqnet_score_mt,peptide_wt,mhcseqnet_score_wt
0,HLA-C*04:01,8,TPRTRNKF,seq14936,0.747080,PPRTRNKF,0.855270
1,HLA-C*04:01,8,SWPGAPVY,seq14939,0.979324,SWPGTPVY,0.970322
2,HLA-C*04:01,8,YVAVCHPL,seq14940,0.762039,YVAVCRPL,0.962414
3,HLA-C*04:01,8,RELLSLAF,seq14941,0.849294,RGLLSLAF,0.765177
4,HLA-C*04:01,8,NQYASVAL,seq14942,0.926395,NaN,NaN
...,...,...,...,...,...,...,...
4942,HLA-C*04:01,11,LHGTLQKDGWT,seq65802,0.999989,LRGTLQKDGWT,0.999874
4943,HLA-C*04:01,11,SSSSSSSNNNY,seq65806,0.995367,SSSSSSNNNNY,0.995421
4944,HLA-C*04:01,11,RSSSSSSSSSN,seq65807,0.995207,RSSSSSSSSNN,0.997723
4945,HLA-C*04:01,11,SSSSSSSSSNN,seq65808,0.999866,SSSSSSSSNNN,0.999945
